In [118]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[0]
sys.path.append(str(ROOT))

# Setup

In [119]:
# reused from 02_baselines.ipynb
import numpy as np
import pandas as pd
from src.data import load_train_data

data = load_train_data()

TARGET = 'SalePrice'

y = np.log1p(data[TARGET])
X = data.drop(columns=[TARGET]) 



In [120]:
# feature grouping, reused from baselines
nominal_cols = ['MSZoning','MSSubClass', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType','Foundation', 'Heating','CentralAir', 'Electrical','Functional',
       'GarageType', 'GarageFinish','PavedDrive','MiscFeature',
       'SaleType', 'SaleCondition','Fence'
]

ordinal_cols = ['OverallQual','OverallCond','ExterQual', 'ExterCond',
                'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
                'HeatingQC','KitchenQual','FireplaceQu','GarageQual',
                'GarageCond','PoolQC'

]
num_structured_missing = ['MasVnrArea','GarageYrBlt']
num_unstruc_missing = ['LotFrontage']

complete_num_cols = [
  col for col in data.columns
  if col not in nominal_cols + ordinal_cols + ["SalePrice"] + num_structured_missing + num_unstruc_missing
]

# check if I got everything
print(len(nominal_cols)+len(ordinal_cols)+len(num_structured_missing)+len(num_unstruc_missing)+len(complete_num_cols) +1)#1 for the saleprice

81


# Feature Engineering

I will use the feature matrix X to conduct feature engineering, data is kept isolated 

It was observed that there are a lot of premium amenities a house can have, such as pool, garage, Fireplace, masonry veneer ...  And certainly not all the houses have them, however, in the dataset, they are all represented by a structured missingness in some attributes, NaN means no such amenity, It will perhaps be beneficial if a binary feature can be engineered to indicate if the house has a certain feature

Previously in EDA, we identified some attributes with significant percentage of missing values, they are actually the rare amentities in the house, I have listed them below:
1. Pool
2. Miscelaneous feature not covered in other categories
3. Alley
4. Fence
5. Masonry Veneer
6. Fireplace

I will start with these six rare amentities

Except from these 6 features, there are also other more common but still premium features: 
1. Garage
2. Basement 

In [121]:
X_v1 = X.copy()

# add binary indicators for amenities 
X_v1['HasPool'] = X_v1['PoolQC'].notna().astype(int) # 1 has pool, 2 no pool
X_v1['HasMisc'] = X_v1['MiscFeature'].notna().astype(int)
X_v1['HasAlley'] = X_v1['Alley'].notna().astype(int)
X_v1['HasFence'] = X_v1['Fence'].notna().astype(int)
X_v1['HasMasVnr'] = (X_v1['MasVnrArea']>0).astype(int)
X_v1['HasFireplace'] = X_v1['FireplaceQu'].notna().astype(int)
X_v1['HasBasement'] = X_v1['BsmtQual'].notna().astype(int)
X_v1['HasGarage'] = X_v1['GarageType'].notna().astype(int)

Previously in EDA, I have calculated the pearson correlation of numeric attributes with the target, and the top features with the highest correlation are Area Features, some of them are:
1. GrLiveArea
2. GarageCars
3. GarageArea
4. TotalBsmtSF

These are all size and area related features, it shows that buyers care more about the area, however there isn't an attribute that summarises the total usable area of the house, and I shall come up with that


In [122]:
X_v1['TotalInHouseArea'] = X_v1['TotalBsmtSF'] + X_v1['1stFlrSF'] + X_v1['2ndFlrSF'] # total area in house



Buyers might care for the age of the house at the point of buying, as older houses mean more wear and tear and risk higher maintainence costs

In [123]:
X_v1['HouseSaleAge'] = X_v1['YrSold'] - X_v1['YearBuilt']

# if a house has got remodelled before, the house might had major fixes and that might matter to the buyers
# so I will add that in as well
X_v1['RemodAge'] = X_v1['YrSold'] - X_v1['YearRemodAdd'] # if no remod, same as house age 

# Encoding

In [124]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

Numerical attributes encoding - reused from baselines

In [125]:
# for numerical attributes with structured missingness, value 0 will be imputed 
num_struc_missing_pipeline = Pipeline([
  ('imputer',
   SimpleImputer(strategy='constant',
   fill_value = 0)),
  ('scaler',StandardScaler())
])


# for numerical attributes with true missingness, the median value will be imputed 
num_unstruc_missing_pipeline = Pipeline([
  ('imputer',
   SimpleImputer(strategy='median')),
   ('scaler',StandardScaler())
])

other_numerical_cols = [
  col for col in data.columns
  if col not in nominal_cols + ordinal_cols + ["SalePrice"] + num_structured_missing + num_unstruc_missing
]

# dummy imputer, since no missing data is expected 
other_num_pipeline = Pipeline([
  ('imputer',
   SimpleImputer(strategy='median')),
   ('scaler',StandardScaler())
])

ordinal encoding - reused from EDA

In [126]:
ordinal_cols = ['OverallQual','OverallCond','ExterQual', 'ExterCond',
                'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
                'HeatingQC','KitchenQual','FireplaceQu','GarageQual',
                'GarageCond','PoolQC'
]

no_structured_missing = ['OverallQual','OverallCond','ExterQual', 'ExterCond',
                         'HeatingQC','KitchenQual']

structured_missing = [col for col in ordinal_cols if col not in no_structured_missing]

In [127]:
# in the copied dataset, change structured missingness from NaN to a meaningful None
for col in structured_missing:
  X_v1[col] = X_v1[col].fillna("None")

In [128]:
# Previously, I have already filled the NaNs of ordinal attributes with None, where applicable 
Qual_and_Con_scale = ['None','Po','Fa','TA','Gd','Ex']
Exposure_scale = ['None','No','Mn','Av','Gd']
Fintype_scale = ['None','Unf','LwQ','Rec','BLQ','ALQ','GLQ']


# define the columns that uses each scale 
qnc_cols = ['ExterQual','ExterCond','BsmtQual','BsmtCond','HeatingQC','KitchenQual','FireplaceQu',
'GarageQual','GarageCond','PoolQC']
exp_col = ['BsmtExposure']
fintype_cols = ['BsmtFinType1','BsmtFinType2']


# Generate mappings
qnc_map = {k: i for i, k in enumerate(Qual_and_Con_scale)}
exp_map = {k: i for i, k in enumerate(Exposure_scale)}
fintype_map = {k: i for i, k in enumerate(Fintype_scale)}


# apply the mappings
for col in qnc_cols:
  X_v1[col] = X_v1[col].map(qnc_map)

for col in exp_col:
  X_v1[col] = X_v1[col].map(exp_map)

for col in fintype_cols:
  X_v1[col] = X_v1[col].map(fintype_map)

Since Columntransformer does not pass through columns unless it is told to, I will add a pipeline that does practically nothing for the sake of adding my ordinal encoding into the pipeline

In [129]:
ordinal_pipeline = Pipeline([
  ('imputer', SimpleImputer(strategy='most_frequent'))
])

# does practically nothing because all the missingness for ordinal columns have already been handled

# Combined pipelines

In [130]:
preprocessor = ColumnTransformer([
  ('num_struc',num_struc_missing_pipeline,
    num_structured_missing),
  ('num_unstruc',num_unstruc_missing_pipeline,
    num_unstruc_missing),
  ('other_num', other_num_pipeline, 
   other_numerical_cols),
  ('nominal', OneHotEncoder(handle_unknown='ignore'), nominal_cols),
  ('ordinal',ordinal_pipeline,ordinal_cols)
])

# Models & Training

In [131]:
model = Ridge(alpha=1.0)

pipeline = Pipeline([
  ('prep', preprocessor),
  ('model', model)
])

scores = cross_val_score(
  pipeline, X_v1, y,
  scoring= 'neg_root_mean_squared_error',
  cv = 5
)

rmse = -scores.mean()
print(rmse)

0.1425581059397415


## Model RMSE Log

Benchmark RMSE from baseline: 0.147378
| versions | changes | RMSE | remarks |
|----------|---------|------|---------|
| v1| added engineered features | 0.14255 | raw features remained in the dataset|
| v2| aggressively removed features that are relevant to the already engineered features| 0.14114| NA|

# Ablation study on engineered features

In [132]:
X_eng = X.copy()

# keep engineered features

X_eng['HasPool'] = X_eng['PoolQC'].notna().astype(int) # 1 has pool, 2 no pool
X_eng['HasMisc'] = X_eng['MiscFeature'].notna().astype(int)
X_eng['HasAlley'] = X_eng['Alley'].notna().astype(int)
X_eng['HasFence'] = X_eng['Fence'].notna().astype(int)
X_eng['HasMasVnr'] = (X_eng['MasVnrArea']>0).astype(int)
X_eng['HasFireplace'] = X_eng['FireplaceQu'].notna().astype(int)
X_eng['HasBasement'] = X_eng['BsmtQual'].notna().astype(int)
X_eng['HasGarage'] = X_eng['GarageType'].notna().astype(int)

X_eng['TotalInHouseArea'] = X_eng['TotalBsmtSF'] + X_eng['1stFlrSF'] + X_eng['2ndFlrSF'] # total area in house

X_eng['HouseSaleAge'] = X_eng['YrSold'] - X_eng['YearBuilt']

X_eng['RemodAge'] = X_eng['YrSold'] - X_eng['YearRemodAdd']

# some related features were also removed, since now I only care if the house have such feature
X_eng = X_eng.drop(
  columns=[
    'PoolQC','MiscFeature','Alley','Fence','MasVnrArea','FireplaceQu',
    'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2',
    'GarageType', 'GarageQual', 'GarageCond',
    'TotalBsmtSF','1stFlrSF','2ndFlrSF',
    'YrSold','YearBuilt','YearRemodAdd'
  ]
)

# remove features that were removed
nominal_cols = ['MSZoning','MSSubClass', 'Street',  'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType','Foundation', 'Heating','CentralAir', 'Electrical','Functional',
        'GarageFinish','PavedDrive',
       'SaleType', 'SaleCondition'
]

ordinal_cols = ['OverallQual','OverallCond','ExterQual', 'ExterCond',
                'HeatingQC','KitchenQual'

]
num_structured_missing = ['GarageYrBlt']
num_unstruc_missing = ['LotFrontage']

rest_num_cols = [
  col for col in X_eng.columns
  if col not in nominal_cols + ordinal_cols + ["SalePrice"] + num_structured_missing + num_unstruc_missing
]

# pipelines

# for numerical attributes with structured missingness, value 0 will be imputed 
num_struc_missing_pipeline = Pipeline([
  ('imputer',
   SimpleImputer(strategy='constant',
   fill_value = 0)),
  ('scaler',StandardScaler())
])

# for numerical attributes with true missingness, and the rest of the num cols, the median value will be imputed 
rest_num = Pipeline([
  ('imputer',
   SimpleImputer(strategy='median')),
   ('scaler',StandardScaler())
])

# Only few columns need encoding now 
Qual_and_Con_scale = ['None','Po','Fa','TA','Gd','Ex']
qnc_cols = ['ExterQual','ExterCond','HeatingQC','KitchenQual']

# Generate mappings
qnc_map = {k: i for i, k in enumerate(Qual_and_Con_scale)}

# apply the mappings
for col in qnc_cols:
  X_eng[col] = X_eng[col].map(qnc_map)

# impute ordinal (dummy)
ordinal_pipeline = Pipeline([
  ('imputer', SimpleImputer(strategy='most_frequent'))
])

# preprocessor
preprocessor = ColumnTransformer([
  ('num_struc',num_struc_missing_pipeline,
    num_structured_missing),
  ('rest_num', rest_num, 
   rest_num_cols),
  ('nominal', OneHotEncoder(handle_unknown='ignore'), nominal_cols),
  ('ordinal',ordinal_pipeline,ordinal_cols)
])


# model
model = Ridge(alpha=1.0)

pipeline = Pipeline([
  ('prep', preprocessor),
  ('model', model)
])

scores = cross_val_score(
  pipeline, X_eng, y,
  scoring= 'neg_root_mean_squared_error',
  cv = 5
)

rmse = -scores.mean()
print(rmse)



0.14113952679467384
